# 🧪 ETL Pipeline Prototype (v2): Single Company (AAPL) with Fundamentals
This notebook implements the complete data processing pipeline with **LightGBM** and **Optuna** in mind.

### Transformation List:
1. **Cleaning:** Cast types and forward-fill price gaps.
2. **Fundamental Merging:** Join daily prices with quarterly income/balance data (Point-in-Time alignment).
3. **Normalization:** Calculate **Log-Returns** for stationarity.
4. **Trend Battery:** SMA (20, 50, 200).
5. **Momentum Battery:** RSI (14) and MACD.
6. **Volatility Battery:** Bollinger Bands and ATR.
7. **Targets:** Next-day Log-Return (Regression) and Binary Move (Classification).
8. **EDA Summary:** info(), describe(), and null analysis.

In [1]:
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# 1. Load Data
df_prices = pl.read_parquet("../data/raw/prices.parquet").filter(pl.col("Ticker") == "AAPL").sort("Date")
df_income = pl.read_parquet("../data/raw/income.parquet").filter(pl.col("Ticker") == "AAPL").sort("Report Date")
df_balance = pl.read_parquet("../data/raw/balance.parquet").filter(pl.col("Ticker") == "AAPL").sort("Report Date")
df_meta = pl.read_parquet("../data/raw/companies.parquet").filter(pl.col("Ticker") == "AAPL")

print(f"Prices: {df_prices.shape}, Income: {df_income.shape}, Balance: {df_balance.shape}")

Prices: (1238, 11), Income: (19, 28), Balance: (19, 30)


## 1. Fundamentals Integration
We use a `join_asof` to align quarterly fundamentals with daily prices without look-ahead bias.

In [2]:
# Prepare fundamentals for joining (use 'Publish Date' to avoid look-ahead bias if available, otherwise 'Report Date')
# SimFin uses 'Publish Date' as the date the info became public.
df_fund = df_income.join(df_balance, on=["Ticker", "Report Date"], suffix="_bal")

# Daily join (prices are primary)
df = df_prices.join_asof(
    df_fund.sort("Publish Date"),
    left_on="Date",
    right_on="Publish Date",
    by="Ticker"
)

print(f"Joined shape: {df.shape}")
df.tail()

Joined shape: (1238, 66)


/var/folders/dn/gclqbh253nl2n8zr7mlqnqs40000gn/T/ipykernel_84495/3153471993.py:6: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df = df_prices.join_asof(


Ticker,SimFinId,Date,Open,High,Low,Close,Adj. Close,Volume,Dividend,Shares Outstanding,SimFinId_right,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),Revenue,Cost of Revenue,Gross Profit,Operating Expenses,"Selling, General & Administrative",Research & Development,Depreciation & Amortization,Operating Income (Loss),Non-Operating Income (Loss),"Interest Expense, Net","Pretax Income (Loss), Adj.",Abnormal Gains (Losses),Pretax Income (Loss),"Income Tax (Expense) Benefit, Net",Income (Loss) from Continuing Operations,Net Extraordinary Gains (Losses),Net Income,Net Income (Common),SimFinId_bal,Currency_bal,Fiscal Year_bal,Fiscal Period_bal,Publish Date_bal,Restated Date_bal,Shares (Basic)_bal,Shares (Diluted)_bal,"Cash, Cash Equivalents & Short Term Investments",Accounts & Notes Receivable,Inventories,Total Current Assets,"Property, Plant & Equipment, Net",Long Term Investments & Receivables,Other Long Term Assets,Total Noncurrent Assets,Total Assets,Payables & Accruals,Short Term Debt,Total Current Liabilities,Long Term Debt,Total Noncurrent Liabilities,Total Liabilities,Share Capital & Additional Paid-In Capital,Treasury Stock,Retained Earnings,Total Equity,Total Liabilities & Equity
str,i64,date,f64,f64,f64,f64,f64,i64,f64,i64,i64,str,i64,str,date,date,date,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,i64,str,i64,str,date,date,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""AAPL""",111052,2025-03-10,235.54,236.16,224.22,227.48,226.51,71451281,null,15022073000,111052,"""USD""",2025,"""Q1""",2024-12-31,2025-01-31,2026-01-30,15081724000,15150865000,124300000000,-66025000000,58275000000,-15443000000,-7175000000,-8268000000,null,42832000000,-248000000,null,42584000000,null,42584000000,-6254000000,36330000000,null,36330000000,36330000000,111052,"""USD""",2025,"""Q1""",2025-01-31,2025-01-31,15081724000,15150865000,53775000000,29639000000,6911000000,133240000000,46069000000,87593000000,77183000000,210845000000,344085000000,61910000000,1995000000,133517000000,94804000000,143810000000,277327000000,84768000000,null,-11221000000,66758000000,344085000000
"""AAPL""",111052,2025-03-11,223.81,225.84,217.45,220.84,219.9,76137410,null,15022073000,111052,"""USD""",2025,"""Q1""",2024-12-31,2025-01-31,2026-01-30,15081724000,15150865000,124300000000,-66025000000,58275000000,-15443000000,-7175000000,-8268000000,null,42832000000,-248000000,null,42584000000,null,42584000000,-6254000000,36330000000,null,36330000000,36330000000,111052,"""USD""",2025,"""Q1""",2025-01-31,2025-01-31,15081724000,15150865000,53775000000,29639000000,6911000000,133240000000,46069000000,87593000000,77183000000,210845000000,344085000000,61910000000,1995000000,133517000000,94804000000,143810000000,277327000000,84768000000,null,-11221000000,66758000000,344085000000
"""AAPL""",111052,2025-03-12,220.14,221.75,214.91,216.98,216.05,62547467,null,15022073000,111052,"""USD""",2025,"""Q1""",2024-12-31,2025-01-31,2026-01-30,15081724000,15150865000,124300000000,-66025000000,58275000000,-15443000000,-7175000000,-8268000000,null,42832000000,-248000000,null,42584000000,null,42584000000,-6254000000,36330000000,null,36330000000,36330000000,111052,"""USD""",2025,"""Q1""",2025-01-31,2025-01-31,15081724000,15150865000,53775000000,29639000000,6911000000,133240000000,46069000000,87593000000,77183000000,210845000000,344085000000,61910000000,1995000000,133517000000,94804000000,143810000000,277327000000,84768000000,null,-11221000000,66758000000,344085000000
"""AAPL""",111052,2025-03-13,215.95,216.84,208.42,209.68,208.78,61368330,null,15022073000,111052,"""USD""",2025,"""Q1""",2024-12-31,2025-01-31,2026-01-30,15081724000,15150865000,124300000000,-66025000000,58275000000,-15443000000,-7175000000,-8268000000,null,42832000000,-248000000,null,42584000000,null,42584000000,-6254000000,36330000000,null,36330000000,36330000000,111052,"""USD""",2025,"""Q1""",2025-01-31,2025-01-31

## 2. Technical Indicator Battery (Polars)
We define our battery of normalized features.

In [3]:
df = df.with_columns([
    # 1. Normalization (Log Returns)
    (pl.col("Close") / pl.col("Close").shift(1)).log().alias("Log_Return_1d"),
    
    # 2. Trend
    (pl.col("Close") / pl.col("Close").rolling_mean(20) - 1).alias("Dist_SMA_20"),
    (pl.col("Close") / pl.col("Close").rolling_mean(50) - 1).alias("Dist_SMA_50"),
    
    # 3. Momentum (RSI Calculation)
    # Note: Simplified RSI for the prototype
    ((pl.col("Close").diff() > 0).cast(pl.Int32).rolling_mean(14) / 
     (pl.col("Close").diff().abs().rolling_mean(14) + 1e-9)).alias("Relative_Strength"),
    
    # 4. Volatility
    pl.col("Close").rolling_std(20).alias("Vol_20"),
    
    # 5. Fundamental Ratios (if columns exist)
    (pl.col("Net Income") / pl.col("Total Assets")).alias("ROA").fill_null(strategy="forward")
])
print("Features calculated.")

Features calculated.


## 3. Targets (Normalization & Reversion)
We predict the next day's **Log-Return**.

In [4]:
df = df.with_columns([
    pl.col("Log_Return_1d").shift(-1).alias("Target_Log_Return"),
    (pl.col("Close").shift(-1) > pl.col("Close")).cast(pl.Int64).alias("Target_Is_Up")
])

df = df.drop_nulls(subset=["Target_Log_Return", "Dist_SMA_50"])
print(f"Final Dataset Size: {df.shape}")

Final Dataset Size: (1188, 74)


## 4. EDA Summary

In [7]:
print("--- Info ---")
display(df.schema)

print("\n--- Describe ---")
print(df.describe())

print("\n--- Correlation with Target ---")
cols = ["Log_Return_1d", "Dist_SMA_20", "Dist_SMA_50", "ROA", "Target_Log_Return"]
print(df.select(cols).to_pandas().corr()["Target_Log_Return"])

--- Info ---


Schema([('Ticker', String),
        ('SimFinId', Int64),
        ('Date', Date),
        ('Open', Float64),
        ('High', Float64),
        ('Low', Float64),
        ('Close', Float64),
        ('Adj. Close', Float64),
        ('Volume', Int64),
        ('Dividend', Float64),
        ('Shares Outstanding', Int64),
        ('SimFinId_right', Int64),
        ('Currency', String),
        ('Fiscal Year', Int64),
        ('Fiscal Period', String),
        ('Report Date', Date),
        ('Publish Date', Date),
        ('Restated Date', Date),
        ('Shares (Basic)', Int64),
        ('Shares (Diluted)', Int64),
        ('Revenue', Int64),
        ('Cost of Revenue', Int64),
        ('Gross Profit', Int64),
        ('Operating Expenses', Int64),
        ('Selling, General & Administrative', Int64),
        ('Research & Development', Int64),
        ('Depreciation & Amortization', Int64),
        ('Operating Income (Loss)', Int64),
        ('Non-Operating Income (Loss)', Int64),
        


--- Describe ---
shape: (9, 75)
┌────────────┬────────┬──────────┬────────────┬───┬───────────┬──────────┬────────────┬────────────┐
│ statistic  ┆ Ticker ┆ SimFinId ┆ Date       ┆ … ┆ Vol_20    ┆ ROA      ┆ Target_Log ┆ Target_Is_ │
│ ---        ┆ ---    ┆ ---      ┆ ---        ┆   ┆ ---       ┆ ---      ┆ _Return    ┆ Up         │
│ str        ┆ str    ┆ f64      ┆ str        ┆   ┆ f64       ┆ f64      ┆ ---        ┆ ---        │
│            ┆        ┆          ┆            ┆   ┆           ┆          ┆ f64        ┆ f64        │
╞════════════╪════════╪══════════╪════════════╪═══╪═══════════╪══════════╪════════════╪════════════╡
│ count      ┆ 1188   ┆ 1188.0   ┆ 1188       ┆ … ┆ 1188.0    ┆ 1160.0   ┆ 1188.0     ┆ 1188.0     │
│ null_count ┆ 0      ┆ 0.0      ┆ 0          ┆ … ┆ 0.0       ┆ 28.0     ┆ 0.0        ┆ 0.0        │
│ mean       ┆ null   ┆ 111052.0 ┆ 2022-10-30 ┆ … ┆ 4.789602  ┆ 0.06683  ┆ 0.00073    ┆ 0.527778   │
│            ┆        ┆          ┆ 04:01:12.7 ┆   ┆       

## 5. Log-Return to Price Reversion (Formula)
$$ Price_{t+1} = Price_t \times e^{LogReturn_{pred}} $$

In [6]:
current_price = df["Close"].tail(1).item()
mock_pred_return = 0.01 # 1% gain predicted
pred_price = current_price * np.exp(mock_pred_return)
print(f"Current Price: {current_price:.2f}")
print(f"Predicted Return: {mock_pred_return:.4f}")
print(f"Predicted Price: {pred_price:.2f}")

Current Price: 209.68
Predicted Return: 0.0100
Predicted Price: 211.79
